In [0]:
%run ../gold/00_gold_helpers

In [0]:
logger = get_logger("gold_daily_sales")

try:

    logger.info("Starting Gold Daily Sales transformation")

    # ---------------------------------------------------------
    # Read Silver Sales
    # ---------------------------------------------------------

    logger.info("Reading Silver Sales table")

    df = read_table("sales_clean")

    logger.info("Silver Sales table read successfully")

    # ---------------------------------------------------------
    # Remove Technical Columns
    # ---------------------------------------------------------

    logger.info(
        "Removing technical columns: "
        "_ingestion_timestamp, _source_file"
    )

    df = df.drop(
        "_ingestion_timestamp",
        "_source_file"
    )

    logger.info("Technical columns removed")

    # ---------------------------------------------------------
    # Daily Sales Aggregation
    # ---------------------------------------------------------

    logger.info("Aggregating sales by order_date")

    df_agg = (
        df
        .groupBy("order_date")
        .agg(
            count(col("order_id")).alias("total_orders"),

            sum(col("quantity")).alias(
                "total_quantity"
            ),

            sum(
                col("quantity") * col("unit_price")
                - (col("discount") * 100)
            ).alias("total_sales")
        )
    )

    logger.info(
        "Daily sales aggregation completed"
    )
    logger.info('printing schema and data')
    df.printSchema()
    display(df)
    # ---------------------------------------------------------
    # Create Schema
    # ---------------------------------------------------------

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )

    # ---------------------------------------------------------
    # Save Gold Table
    # ---------------------------------------------------------

    logger.info(
        "Saving Gold Daily Sales Summary table"
    )

    save_table(
        df_agg,
        "daily_sales_summary"
    )

    logger.info(
        "Gold Daily Sales Summary table saved successfully"
    )

    logger.info(
        "Gold Daily Sales transformation completed successfully"
    )

except Exception:

    logger.exception(
        "Gold Daily Sales transformation failed"
    )

    raise